# Pocket-Agent Colab Runner
Run cells in order. Replace <YOUR_GITHUB_REPO_URL> first.

In [ ]:
!git clone <YOUR_GITHUB_REPO_URL>
%cd pocket-agent
!pip -q install -r requirements.txt

In [ ]:
import os
cmd = "python data/generate_data.py --out data/train.jsonl --n 2000 --manual data/manual_examples.jsonl"
if os.path.exists("starter/public_test.jsonl"):
    cmd += " --avoid starter/public_test.jsonl"
if os.path.exists("starter/teacher_examples.jsonl"):
    cmd += " --teacher starter/teacher_examples.jsonl"
print(cmd)
!$cmd
if os.path.exists("starter/public_test.jsonl"):
    !python data/check_overlap.py --train data/train.jsonl --public starter/public_test.jsonl

In [ ]:
!python train.py --model_id Qwen/Qwen2.5-0.5B-Instruct --data data/train.jsonl --out models/adapter

In [ ]:
import subprocess
bonus_cmd = [
    "python", "quantize.py",
    "--base", "Qwen/Qwen2.5-0.5B-Instruct",
    "--adapter", "models/adapter",
    "--out", "models/quantized/model.gguf",
    "--quant", "q3_k_s",
    "--max_mb", "250",
]
rc = subprocess.run(bonus_cmd).returncode
if rc != 0:
    print("Falling back to q4_k_m for safer accuracy while still meeting <=500MB gate")
    subprocess.run([
        "python", "quantize.py",
        "--base", "Qwen/Qwen2.5-0.5B-Instruct",
        "--adapter", "models/adapter",
        "--out", "models/quantized/model.gguf",
        "--quant", "q4_k_m",
    ], check=True)
!ls -lh models/quantized/model.gguf

In [ ]:
from inference import run
print(run("weather in Lahore in C", []))
print(run("convert that to euros", []))

In [ ]:
!zip -r submission_artifacts.zip models inference.py README.md demo data/manual_examples.jsonl

In [ ]:
import os
os.environ["MODEL_PATH"] = "models/quantized/model.gguf"
!python demo/app.py